In [1]:
import cv2
import numpy as np
import time
import threading
import tkinter as tk
from tkinter import filedialog, messagebox
import ttkbootstrap as tb
from ttkbootstrap.constants import *

# === Load YOLO ===
net = cv2.dnn.readNet(r"E:\MINIPROJECT\yolov3.weights", r"E:\MINIPROJECT\yolov3.cfg")
with open(r"E:\MINIPROJECT\coco.names", "r") as f:
    classes = [line.strip() for line in f.readlines()]
layer_names = net.getLayerNames()
output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers().flatten()]

vehicle_classes = ["car", "bus", "truck", "motorbike"]

# === Traffic Signal Logic ===
def get_signal_time(vehicle_count):
    return 10 + min(vehicle_count * 2, 30)  # cap green at 40s

# === Vehicle Detection ===
def detect_vehicles(frame):
    height, width = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
    net.setInput(blob)
    outputs = net.forward(output_layers)

    boxes, confidences, class_ids = [], [], []
    for output in outputs:
        for detection in output:
            scores = detection[5:]
            class_id = int(np.argmax(scores))
            confidence = scores[class_id]
            if confidence > 0.5 and classes[class_id] in vehicle_classes:
                center_x = int(detection[0] * width)
                center_y = int(detection[1] * height)
                w = int(detection[2] * width)
                h = int(detection[3] * height)
                x = int(center_x - w / 2)
                y = int(center_y - h / 2)
                boxes.append([x, y, w, h])
                confidences.append(float(confidence))
                class_ids.append(class_id)

    indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)
    vehicle_count = 0

    for i in indexes.flatten():
        x, y, w, h = boxes[i]
        label = classes[class_ids[i]]
        color = (0, 255, 0)
        cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
        cv2.putText(frame, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        vehicle_count += 1

    return vehicle_count

# === Video Processing with GUI Updates ===
def process_video_gui(input_video, output_video, status_label):
    cap = cv2.VideoCapture(input_video)
    if not cap.isOpened():
        messagebox.showerror("Error", "Could not open video file.")
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        vehicle_count = detect_vehicles(frame)
        signal_time = get_signal_time(vehicle_count)
        signal_color = (0, 255, 0) if vehicle_count >= 3 else (0, 0, 255)

        # Annotate Frame
        cv2.putText(frame, f'Vehicles: {vehicle_count}', (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)
        cv2.putText(frame, f'Signal Time: {signal_time}s', (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, signal_color, 2)
        cv2.putText(frame, f'Signal: {"GREEN" if vehicle_count >= 3 else "RED"}', (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 1, signal_color, 2)

        out.write(frame)
        frame_count += 1
        status_label.config(text=f"Processing Frame: {frame_count}")
        cv2.waitKey(1)

    cap.release()
    out.release()
    status_label.config(text="✅ Processing Complete")
    messagebox.showinfo("Done", "Video has been successfully processed!")

# === GUI ===
def start_gui():
    def browse_input():
        path = filedialog.askopenfilename(filetypes=[("MP4 files", "*.mp4")])
        if path:
            input_entry.delete(0, tk.END)
            input_entry.insert(0, path)

    def browse_output():
        path = filedialog.asksaveasfilename(defaultextension=".mp4", filetypes=[("MP4 files", "*.mp4")])
        if path:
            output_entry.delete(0, tk.END)
            output_entry.insert(0, path)

    def start_processing_thread():
        input_path = input_entry.get()
        output_path = output_entry.get()
        if input_path and output_path:
            status_label.config(text="🕒 Processing...")
            thread = threading.Thread(target=process_video_gui, args=(input_path, output_path, status_label), daemon=True)
            thread.start()
        else:
            messagebox.showwarning("Input Required", "Please select both input and output video files.")

    window = tb.Window(themename="superhero")
    window.title("🚦 Adaptive Traffic Signal Controller")
    window.geometry("700x400")
    window.resizable(False, False)

    tb.Label(window, text="🚥 Adaptive Traffic Signal Controller", font=("Segoe UI", 20, "bold")).pack(pady=20)

    input_frame = tb.Frame(window)
    input_frame.pack(padx=20, pady=10, fill="x")
    tb.Label(input_frame, text="🎥 Input Video:").pack(anchor="w")
    input_entry = tb.Entry(input_frame, width=60)
    input_entry.pack(side="left", padx=5, pady=5)
    tb.Button(input_frame, text="Browse", command=browse_input).pack(side="right")

    output_frame = tb.Frame(window)
    output_frame.pack(padx=20, pady=10, fill="x")
    tb.Label(output_frame, text="💾 Output Video:").pack(anchor="w")
    output_entry = tb.Entry(output_frame, width=60)
    output_entry.pack(side="left", padx=5, pady=5)
    tb.Button(output_frame, text="Browse", command=browse_output).pack(side="right")

    status_frame = tb.Frame(window)
    status_frame.pack(padx=20, pady=10, fill="x")
    global status_label
    status_label = tb.Label(status_frame, text="Status: Waiting...", font=("Segoe UI", 10, "italic"))
    status_label.pack(side="left")

    tb.Button(window, text="🚀 Start Processing", command=start_processing_thread, bootstyle="success outline", width=30).pack(pady=20)

    window.mainloop()

# === Run GUI ===
if __name__ == "__main__":
    start_gui()
